In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DIR_OUTPUTS = "../outputs"

In [2]:
recs  = pd.read_csv(f"{DIR_OUTPUTS}/reranked_recommendations.csv")
truth = pd.read_csv(f"{DIR_OUTPUTS}/coupon_response_all_truth.csv")

print(f"Recommendations: {len(recs):,} rows | splits: {recs['split'].unique()}")
print(f"Truth items: {len(truth):,} rows")

Recommendations: 44,760 rows | splits: <StringArray>
['validation', 'test']
Length: 2, dtype: str
Truth items: 18,203 rows


## 1. Quick sanity check on the loaded data

In [5]:
# Events per split
print(recs.groupby("split")["event_id"].nunique().rename("unique events"))

# Positive events: events where at least one item was purchased within 5 days
positive_events = (
    recs[recs["success_within_5d_observed"]]
    .groupby("split")["event_id"]
    .nunique()
    .rename("positive events")
)
print("\n", positive_events)

# Coupon-recommended items and their average discount signal
print("\nAverage discount_signal by recommend_coupon flag:")
print(recs.groupby("recommend_coupon")["discount_signal"].describe())

split
test           715
validation    1523
Name: unique events, dtype: int64

 split
test           68
validation    519
Name: positive events, dtype: int64

Average discount_signal by recommend_coupon flag:
                    count      mean       std  min       25%       50%  \
recommend_coupon                                                         
True              44760.0  0.050859  0.079406  0.0  0.003277  0.017613   

                       75%  max  
recommend_coupon                 
True              0.060888  1.0  


# Business Utility@K Evaluation

**Goal (RQ2/RQ3 from the proposal):** Measure whether the coupon recommendation list is *economically useful* — not just accurate — by computing a Business Utility proxy that balances recommendation hits against estimated discount cost.

## Formula

$$\text{BusinessUtility@K} = \text{Hits@K} - \lambda_{\text{cost}} \times \text{DiscountCost@K}$$

Where:
- **Hits@K** = number of items in the top-K list that the household actually bought within 5 days (binary, 1 per hit)
- **DiscountCost@K** = sum of `discount_signal` for items in top-K that we flagged `recommend_coupon=True`
- **λ** = the discount cost weight; higher λ penalises expensive coupon allocations more

This is the revenue-minus-discount proxy described in the proposal. We do not claim to optimise real profit because the dataset does not include product cost or true margin.

## What `discount_signal` captures

The `discount_signal` column in `reranked_recommendations.csv` encodes the historical ratio of retail discount and coupon discount for a product relative to its household's purchase history. It is already normalised to [0, 1] and was included in the XGBoost feature set (though XGBoost assigned it zero importance, meaning the *ranking* did not use it — but it is still a valid cost proxy for *post-hoc* business utility measurement).

In [ ]:
def business_utility_at_k(df, k, lambda_cost=0.5, split="test"):
    """
    Compute mean Business Utility@K across all events in the given split.

    BU@K (per event) = hits@K  -  lambda_cost * discount_cost@K

    hits@K: number of top-K items where success_within_5d_observed == True
    discount_cost@K: sum of discount_signal for top-K items where recommend_coupon == True
    """
    sub = df[(df["split"] == split) & (df["rank"] <= k)].copy()
    sub["hit_value"]      = sub["success_within_5d_observed"].astype(float)
    sub["discount_cost"]  = sub["discount_signal"] * sub["recommend_coupon"].astype(float)

    per_event = sub.groupby("event_id").agg(
        hits          = ("hit_value",     "sum"),
        discount_cost = ("discount_cost", "sum")
    )
    per_event["bu"] = per_event["hits"] - lambda_cost * per_event["discount_cost"]
    return per_event["bu"].mean()


def business_utility_table(df, k_values=(5, 10, 20), lambdas=(0.0, 0.5, 1.0, 2.0), split="test"):
    rows = []
    for k in k_values:
        for lam in lambdas:
            bu = business_utility_at_k(df, k=k, lambda_cost=lam, split=split)
            rows.append({"K": k, "lambda": lam, "BU@K": round(bu, 4)})
    return pd.DataFrame(rows).pivot(index="K", columns="lambda", values="BU@K")